In [1]:
import os
import sys

current_path = os.path.abspath(os.getcwd())
base_path = os.path.abspath(os.path.join(current_path, '../../'))
sys.path.append(base_path)

import tensorflow as tf
from tensorflow.keras.callbacks import TensorBoard

import numpy as np
from tqdm import tqdm

from data_loader import DataLoader
import project_images_loader as pil

import discriminator as Discriminator
import generator as Generator

import util.functions as fc
import util.util_deep_learning as udl




2024-02-29 18:03:31.585064: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-02-29 18:03:36.087232: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-02-29 18:03:36.088126: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-02-29 18:03:36.309490: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-02-29 18:03:36.879574: I tensorflow/core/platform/cpu_feature_guar

### Loss functions

In [2]:
cross_entropy = tf.keras.losses.BinaryCrossentropy()

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)


In [3]:
# @tf.function
# def val_step(model, x_images, y_true, masks, class_centers, epoch):
#     y_pred, x_feat, x_gap = model(x_images, training=False)
#     x_weights = model.layers[-1].weights[0]
#     loss, loss_ce, loss_cskl, loss_cl, loss_cskl_mask = combined_loss(y_true, y_pred, x_feat, x_weights, masks, x_gap, class_centers, epoch)
#     return y_pred, loss, loss_ce, loss_cskl, loss_cl, loss_cskl_mask

@tf.function
def train_step(G, D, x_batch, y_batch, g_optimizer, d_optimizer):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        y_batch_fake = G(x_batch, training=True)

        real_output = D(y_batch, training=True)
        fake_output = D(y_batch_fake, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, G.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, D.trainable_variables)

    g_optimizer.apply_gradients(zip(gradients_of_generator, G.trainable_variables))
    d_optimizer.apply_gradients(zip(gradients_of_discriminator, D.trainable_variables))

    return y_batch_fake, gen_loss, disc_loss



In [4]:
# data_train = DataLoader(f"{pil.get_base_folder()}data/data_reduced/train", batch_size=1, image_size=(200, 240, 240), shuffle=True, channels_first=False)
# data = data_train[0]

# # dloader = DataLoader(root_folder=f"data_test", batch_size=3, image_size=(200, 240, 240))
# fc.imgshow_3D(f"Original", data[0][0,...,0], current_plane=150)

### Training

In [5]:

def train(num_epochs, batch_size):
    input_shape = (200, 240, 240)

    # ---- Models
    D = Discriminator.create_discriminator_model(input_shape)
    G = Generator.create_generator_model(input_shape)

    # G.load_weights(f"{pil.get_base_folder()}MPGAN-master/weights/generator{9}.h5")
    # D.load_weights(f"{pil.get_base_folder()}MPGAN-master/weights/discriminator{9}.h5")

    # optimizers
    inital_lr = 1e-4
    g_optimizer = tf.keras.optimizers.Adam(inital_lr)
    d_optimizer = tf.keras.optimizers.Adam(inital_lr)

    # ---- Data
    data_train = DataLoader(f"{pil.get_data_reduced_folder(absolute=True)}/train", batch_size=1, image_size=input_shape, channels_first=False)
    # data_val = DataLoader(f"{pil.get_base_folder()}data_test/val", batch_size=batch_size, image_size=input_shape, channels_first=False)


    # visualization
    nb_loss = 2
    acum_loss_train = udl.AcumulativeMetrics(nb_metrics = nb_loss)
    tensorboard_callback = TensorBoard(log_dir=f"{current_path}/models/logs")

    for epoch in range(0,num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")

        # ---- Train
        progress_bar = tqdm(total=len(data_train), desc=f'Initializing...')
        for step, (x_batch, y_batch) in enumerate(data_train):

            y_batch_fake, gen_loss, disc_loss = train_step(G, D, x_batch, y_batch, g_optimizer, d_optimizer)
            acum_loss_train.update((gen_loss, disc_loss))
            loss_mean = acum_loss_train.get_mean()
            
            # img_sim = metrics.compute_ssim(y_batch, y_batch_fake)
            img_sim = 0.0
            progress_bar.set_description(desc=f'Sim: {img_sim:.5f} \t gen_loss: {loss_mean[0]:.5f} \t disc_loss: {loss_mean[1]:.5f}')
            progress_bar.update(1)

            with tf.summary.create_file_writer(tensorboard_callback.log_dir).as_default():
                actual_step = epoch*len(data_train)+step
                layer = 10

                tf.summary.scalar('generator_loss', gen_loss, step=actual_step)
                tf.summary.scalar('discriminator_loss', disc_loss, step=actual_step)
                tf.summary.image("Input", udl.convert_image_to_tensor(x_batch[0, :,:,layer, :]), step=actual_step)
                tf.summary.image("Expected", udl.convert_image_to_tensor(y_batch[0, :,:,layer, :]), step=actual_step)
                tf.summary.image("Generated", udl.convert_image_to_tensor(y_batch_fake[0, :,:,layer, :]), step=actual_step)
                
        del progress_bar
        acum_loss_train.reset()

        # ---- End Train

        if epoch % 5 == 0:
            G.save_weights(f"{current_path}/models/generator{epoch}.h5")
            D.save_weights(f"{current_path}/models/discriminator{epoch}.h5")

        # # ---- Validation
        # calculate_accuracy_val.reset_states()
        # progress_bar = tqdm(total=val_data.steps_per_epoch, desc=f'Initializing...')
        # for step, ((x_images, masks), y_true) in enumerate(val_data.generate()):
        #     if step >= val_data.steps_per_epoch:
        #         break

        #     y_pred, loss, loss_ce, loss_cskl, loss_cl, loss_cskl_mask =  val_step(model, x_images, y_true, masks, cle.class_centers, epoch)

        #     acuculative_loss_val.update((loss, loss_ce, loss_cskl, loss_cl, loss_cskl_mask))
        #     loss_mean = acuculative_loss_val.get_mean()
        #     calculate_accuracy_val.update_state(y_true, y_pred)

        #     progress_bar.set_description(desc=f'Acc: {calculate_accuracy_val.result():.5f} \t loss: {loss_mean[0]:.5f} \t l_ce: {loss_mean[1]:.5f} \t l_cskl: {loss_mean[2]:.5f} \t l_cl: {loss_mean[3]:.5f} \t l_csklm: {loss_mean[4]:.5f}')
        #     progress_bar.update(1)

        # del progress_bar
        # # ---- End Validation



        # # ---- Visualization
        # Update TensorBoard logs

        # # save history and current model
        # acc = np.array((calculate_accuracy_train.result(), calculate_accuracy_val.result()))
        # hist = np.concatenate((acc, acuculative_loss_train.get_mean(), acuculative_loss_val.get_mean()), dtype=np.float32)
        # history_saver.update_and_save(hist)

        # # early stopping and save early model
        # model.save_weights(pil.get_model_path_name(f'e{epoch}_{model_prefix}_{model_name}.h5'))
        # if early_stopping.on_epoch_end(calculate_accuracy_val.result(), model=model):
        #     break




train(num_epochs=101, batch_size=1)

Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


2024-02-29 18:08:20.622959: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79086 MB memory:  -> device: 0, name: NVIDIA A100 80GB PCIe, pci bus id: 0000:8a:00.0, compute capability: 8.0



Epoch 1/101


Initializing...:   0%|          | 0/8 [00:00<?, ?it/s]2024-02-29 18:08:41.010232: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2024-02-29 18:08:43.585542: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-02-29 18:08:52.938892: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-02-29 18:09:25.231276: I external/local_xla/xla/service/service.cc:168] XLA service 0x7fb2a323ab90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-02-29 18:09:25.231310: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA A100 80GB PCIe, Compute Capability 8.0
2024-02-29 18:09:25.952993: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0


Epoch 2/101


...

Sim: 0.00000 	 gen_loss: 1.33604 	 disc_loss: 0.72438: 100%|██████████| 8/8 [00:12<00:00,  1.57s/it]



Epoch 100/101


Sim: 0.00000 	 gen_loss: 1.50772 	 disc_loss: 0.73173: 100%|██████████| 8/8 [00:12<00:00,  1.57s/it]



Epoch 101/101


Sim: 0.00000 	 gen_loss: 1.34378 	 disc_loss: 1.36283: 100%|██████████| 8/8 [00:12<00:00,  1.57s/it]


In [6]:
# Sim: 0.00000 	 gen_loss: 0.93415 	 disc_loss: 1.11709: 100%|██████████| 8/8 [00:14<00:00,  1.86s/it]
# Sim: 0.00000 	 gen_loss: 1.22538 	 disc_loss: 1.64933: 100%|██████████| 8/8 [00:14<00:00,  1.86s/it]